<h1 style="font-size: 40px; margin-bottom: 0px;">5.2 Python image analysis (II)</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 800px;"></hr>

In notebook 5-1, we started exploring how images are represented on our computers by playing around with them in Python. Today, we'll take what we know about how images are represented to extract quantitative information from our images and use that information to analyze our images to derive biological insights. First, we'll load in the single set of serum-starved and serum-stimulated cells that we worked with in in notebook 5-1, then run through an analysis to quantify the nuclear intensity of YAP after serum stimulation. Then, you'll use what you learned to determine if there is a significant difference in YAP nuclear localization. 

<strong>Learning objectives:</strong>
<ul>
    <li>Continue exploring images in Python</li>
    <li>Extract quantitative information from images</li>
    <li>Analyze a single image</li>
    <li>Run a quick analysis to compare serum starved and serum stimulated conditions</li>
</ul>

In [ ]:
#Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import os
import skimage as ski
import scipy.ndimage as ndi

<h1 style="font-size: 40px; margin-bottom: 0px;">Extract quantitative information from images</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 850px;"></hr>

To get us started for today's lesson, let's copy over our code to load in our image files in our <code>data</code> subdirectory like we did in notebook 5-1.

<h2>Plot the intensity profile</h2>

Since our images are 2D arrays, we can pull out rows and columns and then plot the resulting intensities along that axis as a line plot to visualize the intensity profile. Select an image from the ones that we've imported, and then see if you can plot an intensity profile by pulling out the intensity values for just a single row or a single column from that image.

<h2>Detect edges</h2>

The quantitative values of our images can be used to determine where edges exist in our images, and we can make use of scikit-image's <code>ski.feature.canny()</code> edge detector function to identify where the edges are. <a href="https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.canny" rel="noopener noreferrer"><u>Documentation for <code>ski.feature.canny()</code> can be found here.</u></a>

This function allows us to set the thresholds for what is considered an edge as well as smooth out noise if our images are noisy, resulting in edges being detected where they don't actually exist.

<h2>Identify particles in an image</h2>

Like with ImageJ, we can also create a binary image to then analyze the properties of the particles selected by our threshold. To do this, let's first process our two DAPI images again to convert our image into a binary image by thresholding. Use the same threshold for both images, so that we are processing both sample conditions in the same way. If there are holes left after thresholding, see if you remember how to fill holes in a binary image from notebook 5-1.

Take a look at both thresholded DAPI images side by side.

<h3>Remove particles on the image edge</h3>

Sometimes it can be helpful to also remove particles that are clipped by the boundaries of our image so that we exclude them from our analysis. In one of our images, we can see that there is a nucleus that is right on the edge. This could potentially throw off our results since we're not capturing the full nucleus. We can remove this particle by using <code>ski.segmentation.clear_border()</code>, which will remove particles that tough the edges of our image and is similar to what you can also do in ImageJ. <a href="https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.clear_border" rel="noopener noreferrer"><u>Documentation for <code>ski.segmentation.clear_border()</code> can be found here.</u></a>

Once we have our processed binaries, we can use scikit-image's <code>ski.measure.label()</code> function to identify and also label individual particles in our thresholded images. <a href="https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.label" rel="noopener noreferrer"><u>Documentation for <code>ski.measure.label()</code> can be found here.</u></a>

Digging into the documentation, we can see that this function works by assigning any element in our 2D matrix with the value <code>0</code> as background pixels by default, and then it identifies particles as clusters of connected elements that share the same value, which in our case is <code>1</code>. Then after it identifies particles, it will assign each particle its own integer label.

If we dig into the documentation, we can see that it returns to us one object by default (<code>labels</code>), which is the labeled array where each particle is assigned an integer value. We can also instruct it to return an additional object (<code>num</code>), which is just the total particle count if we switch the parameter <code>return_num</code> from <code>False</code> to <code>True</code>.

Now let's take a look at the two objects it's returned to us. How many particles do you see when looking at the number of particles compared to how many particles you see when you visualize the returned array using <code>plt.imshow()</code>?

<h3>Visualize labeled particles more distinctly</h3>

We can see that the particles are visualized with slightly different color if we use the viridis colormap because their labels are treated as intensity values. But we can make their colors more distinct from one another based on their labels. 

To do this, we can assign a color to each particle based on their label using <code>ski.color.label2rgb()</code>, which will allow us to more easily differentiate each particle based on their assigned color rather than treating their label as an intensity value. <a href="https://scikit-image.org/docs/stable/api/skimage.color.html#skimage.color.lab2rgb" rel="noopener noreferrer"><u>Documentation for <code>ski.color.label2rgb()</code> can be found here.</u></a>

<h2>Analyze particles</h2>

With our labeled particles, we can continue to make use of scikit-image to now analyze the labeled particles. To do this, we'll use the <code>ski.measure.regionprops()</code> function, which will measure a bunch of properties of each labeled particle. <a href="https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops" rel="noopener noreferrer"><u>Documentation for <code>ski.measure.regionprops()</code>, including the full set of properties that it measures, can be found here.</u></a>

Let's use this function to analyze our labeled particles from our serum and no serum conditions, and then save each resulting output to a new variable. 

Now let's take a look at the resulting object that we have as a result of this function:

We can see that we were returned a compound data type in which each element within the list is a RegionProperties object corresponding to a labeled particle that was analyzed. If we dig into the documentation for <code>ski.measure.regionprops()</code>, we can see under Notes that there are a bunch of properties that can be access as attributes or keys. These include <code>area</code> and <code>label</code>, which are most relevant to our analysis today.

Let's pull out these two attributes for a random particle that we analyzed to take a look.

<h3>Pull particle areas</h3>

You can see that we're able to access the quantified area of each particle by the <code>area</code> attribute, and if we wanted to then pull together all the areas of our analyzed particles, we can iterate through our list of <code>RegionProperties</code> objects that was outputed by the <code>ski.measure.regionprops()</code> function, and then specifically pull out the <code>area</code> attribute from the object.

Let's take a look at the measured areas for each of our particles as a single DataFrame that holds the data for both serum and no serum conditions.

<h3>Filter out noise</h3>

We can see that there are a lot of particles with just a size of 1 or are otherwise very small. This tells us that potentially, our thresholding picked up some noise as well. Let's take a look at the distribution of our particle areas.

You can see that we have a fair number of tiny tiny particles with an area of less than 100 sq pixels, which probably correspond to noise (since we wouldn't expect our nuclei to be that small). So this isn't something we want to include in our analysis. So then we need to filter out these particles to remove them from our analysis. We can make use of a conditional statement to filter out these particles based on whether or not their area meets our threshold to not be considered noise.

For those of you working aheaad, what this essentially looks like is that we need to first set up a new "blank" image using <code>np.zeros_like()</code>. And then we will iterate through each <code>RegionProperties</code> object to transfer any particle whose measured <code>area</code> is above our threshold of 100 pixels over to our "blank" image. We will then end up with a new image containing only particles that are greater than our size threshold.

Let's take a look at our images after we filtered out the noise.

Now that we've filtered out any noise, we can then relabel our particles again using <code>ski.measure.label()</code>.

<h1 style="font-size: 40px; margin-bottom: 0px;">Separate particles using the watershed method</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 850px;"></hr>

For one of our images, we can see that we have two nuclei that are close together, and as a result, they end up getting labeled together as well. We can separate out these nuclei so that they are instead understood as two separate particles rather than a single particle. There are a number of different ways to computationally separate out particles, and we'll be using the watershed method, which is available as a function in scikit-image <code>ski.segmentation.watershed()</code>. <a href="https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.watershed" rel="noopener noreferrer"><u>Documentation for <code>ski.segmentation.watershed()</code> can be found here.</u></a>

First, let's take a look at our filtered and relabeled image for our no serum condition. You should see that there are two nuclei close enough together that they are picked up as a single particle.

We'll first need to prepare our filtered and relabeled image for watershed by dilating our particles so that they are smoother, so the watershed function doesn't mistake small irregularities as things to segment.

To apply a dilation to smooth out irregularities, we can make use of the <code>ski.morphology.dilation()</code> function that we used in notebook 5-1. There, we used this function as part of a process to fill holes, but here, we'll use it to smooth out the edges of our thresholded nuclei.

Then we can calculate the centers of our individual particles by calculating the distance within the particle from the edges (or background), which will give us an idea of where the center of each "true" particle should be. To do this, we'll make use of the <code>ndi.distance_transform_edt()</code> function, which calculates the Euclidean distance from the background. <a href="https://docs.scipy.org/doc//scipy-1.8.0/reference/generated/scipy.ndimage.distance_transform_edt.html" rel="noopener noreferrer"><u>Documentation for <code>ndi.distance_transform_edt()</code> can be found here.</u></a>

For those of you working ahead, pass your dilated image to the <code>ndi.distance_transform_edt()</code> function using default parameters and save the output to a new variable.

Let's take a look to see how the distance map looks like.

You can see that the regions of greatest distance from the background now have the highest value, which we can more readily visualize with the viridis colormap. 

We can then identify the coordinates of the centers of each "true" particle by identifying where the local maxima are using the calculated distances. We'll make use of the <code>ski.feature.peak_local_max()</code> function, which will find the spots of highest value (peaks) and return their coordinates to us. <a href="https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.peak_local_max" rel="noopener noreferrer"><u>Documentation for <code>ski.feature.peak_local_max()</code> can be found here.</u></a>

For those of you working ahead, you'll need to pass your distance map to the <code>ski.feature.peak_local_max()</code> function and then pass an argument for the footprint, which determines the size of the area in which to compare the values to determine the local peak. Save the output to a variable.

How does the output look like?

The output is essentially the coordinates corresponding to where our calculated local peaks were determined to be. We can take these coordinates and set them as points in a 2D matrix matching the shape of our labeled nuclei image. These coordinates will determine where in our image we want to initiate our watershed method.

We can then add labels to each coordinate point.

With those coordinates set and labeled, we can then initiate the watershed using the <code>ski.segmentation.watershed()</code> function. <a href="https://scikit-image.org/docs/0.25.x/api/skimage.segmentation.html#skimage.segmentation.watershed" rel="noopener noreferrer"><u>Documentation for <code>ski.segmentation.watershed()</code> can be found here.</u></a>.

What we're doing is essentially identifying a local minimum, which we can set as the inverse of our local maxima, and then "flood" the region. We can also specify how much "flooding" we want to happen by using our original labeled nuclei as a mask.

The output from this function will be a new image with particles segmented according to the watershed method and using the coordinates and distances we calculated. Save the output to a new variable.

How does the resulting image look like now that you've segmented your nuclei?

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #1: Pull out a single cell</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 850px;"></hr>

Now that we have our nuclei individually labeled, see if you can pull out a single nucleus and display a binary image of your single nucleus.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #2: Measure mean nuclear fluorescence</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 900px;"></hr>

Now that you're able to pull out a single nucleus at a time, see if you can then quantify the mean nuclear fluorescence for a single nucleus. Recall from 5-1 how you can use binary images as a mask, which will allow you to focus on just a particular region of interest, and you can apply this mask to the YAP channel.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #3: Analyze all nuclei in a single image</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 900px;"></hr>

See if you can set up a way to instead of manually analyzing a single nucleus one at a time, to analyze all the nuclei in a single image. Specifically for this exercise, see if you can analyze the mean nuclear YAP fluorescence intensity for all the nuclei in a single image, either the serum-starvation or the serum-stimulation condition.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #4: Perform statistical analysis</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 900px;"></hr>

Let's run a quick statistical analysis on our results from our single set of images.

<h1 style="font-size: 40px; margin-bottom: 0px;">Challenge #1: Analyze nuclear:cytoplasmic ratio</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 900px;"></hr>

In the above examples, we looked at just the nuclear intensity and compared the intensities between our two conditions. Recall back in MCB201A, we analyzed the nuclear:cytoplasmic ratio in order to determine if there was a quantifiable difference in localization and to normalize for differences in signal intensity between different cells.

For those of you who want a challenge, see if you can take what you now know about analyzing images and apply it to the two images that we've been working with to determine if there is a significant difference in the nuclear:cytoplasmic ratio between serum-starved and serum-treated cells. Is there a way you could potentially make it more automated?